# ETL — Отчёты из звёздной схемы → ClickHouse

**Предварительное условие:** ноутбук `01_etl_star_schema.ipynb` уже выполнен.

**6 витрин данных:**
1. `mart_products`        — продажи по продуктам
2. `mart_customers`       — продажи по клиентам
3. `mart_time`            — тренды по времени
4. `mart_stores`          — эффективность магазинов
5. `mart_suppliers`       — эффективность поставщиков
6. `mart_product_quality` — качество продуктов

**Стратегия записи:** Spark выполняет агрегации, результат (небольшой DataFrame)  
конвертируется в pandas и загружается через `clickhouse-connect`.

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import clickhouse_connect

## 1. SparkSession

In [8]:
spark = (
    SparkSession.builder
    .appName("ETL ClickHouse Reports")
    .master("local[*]")
    .config("spark.jars", "/opt/jars/postgresql-42.7.3.jar")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.1.1


## 2. Подключения

In [9]:
PG_URL   = "jdbc:postgresql://postgres:5432/postgres-petshop"
PG_PROPS = {
    "user":     "postgres",
    "password": "postgres",
    "driver":   "org.postgresql.Driver",
}

def read_pg(table: str):
    return spark.read.jdbc(PG_URL, table, properties=PG_PROPS)

# ClickHouse клиент
ch = clickhouse_connect.get_client(
    host="clickhouse", port=8123,
    username="clickhouse", password="clickhouse",
)
print("ClickHouse ping:", ch.ping())

ClickHouse ping: True


## 3. Загрузка звёздной схемы из PostgreSQL

In [10]:
fact         = read_pg("public.fact_sales")
dim_product  = read_pg("public.dim_product")
dim_category = read_pg("public.dim_category")
dim_customer = read_pg("public.dim_customer")
dim_seller   = read_pg("public.dim_seller")
dim_store    = read_pg("public.dim_store")
dim_supplier = read_pg("public.dim_supplier")
dim_location = read_pg("public.dim_location")

# Регистрируем временные представления для Spark SQL
fact.createOrReplaceTempView("fact_sales")
dim_product.createOrReplaceTempView("dim_product")
dim_category.createOrReplaceTempView("dim_category")
dim_customer.createOrReplaceTempView("dim_customer")
dim_seller.createOrReplaceTempView("dim_seller")
dim_store.createOrReplaceTempView("dim_store")
dim_supplier.createOrReplaceTempView("dim_supplier")
dim_location.createOrReplaceTempView("dim_location")

print("fact_sales rows:", fact.count())

fact_sales rows: 10000


## 4. Вспомогательная функция записи в ClickHouse

In [11]:
def write_to_clickhouse(df, table_name: str, ddl: str):
    """DROP + CREATE + INSERT в ClickHouse через clickhouse-connect."""
    ch.command(f"DROP TABLE IF EXISTS reports.{table_name}")
    ch.command(ddl)
    pdf = df.toPandas()
    ch.insert_df(f"reports.{table_name}", pdf)
    print(f"  ✓ reports.{table_name}: {len(pdf)} строк")

---

## Витрина 1: mart_products — продажи по продуктам

**Цель:** выручка, количество продаж, рейтинг по каждому продукту и категории.

In [12]:
mart_products = spark.sql("""
    SELECT
        p.product_id,
        p.product_name,
        c.category_name,
        CAST(SUM(f.quantity) AS BIGINT)         AS total_quantity,
        ROUND(CAST(SUM(f.total_price) AS DOUBLE), 2) AS total_revenue,
        ROUND(COALESCE(ANY_VALUE(p.rating), 0), 2) AS product_rating,
        COALESCE(ANY_VALUE(p.reviews), 0)        AS total_reviews,
        COUNT(f.sale_id)                        AS sale_count
    FROM fact_sales f
    JOIN dim_product  p ON f.product_id  = p.product_id
    JOIN dim_category c ON p.category_id = c.category_id
    GROUP BY p.product_id, p.product_name, c.category_name
    ORDER BY total_revenue DESC
""")

mart_products.show(5)

write_to_clickhouse(
    mart_products,
    "mart_products",
    """
    CREATE TABLE reports.mart_products (
        product_id     Int64,
        product_name   String,
        category_name  String,
        total_quantity Int64,
        total_revenue  Decimal(18, 2),
        product_rating Decimal(3, 1),
        total_reviews  Int32,
        sale_count     Int64
    ) ENGINE = MergeTree()
    ORDER BY (category_name, product_id)
    """,
)

+----------+------------+-------------+--------------+-------------+--------------+-------------+----------+
|product_id|product_name|category_name|total_quantity|total_revenue|product_rating|total_reviews|sale_count|
+----------+------------+-------------+--------------+-------------+--------------+-------------+----------+
|      8455|    Dog Food|          Toy|             8|       862.83|           4.3|           98|         2|
|      9487|    Dog Food|          Toy|             8|       860.82|           1.5|          383|         2|
|      5896|     Cat Toy|          Toy|            16|       819.92|           4.2|          383|         2|
|      1945|   Bird Cage|          Toy|             7|       775.06|           1.3|          350|         2|
|      4692|     Cat Toy|          Toy|            14|       768.08|           4.6|          977|         2|
+----------+------------+-------------+--------------+-------------+--------------+-------------+----------+
only showing top 5 

---

## Витрина 2: mart_customers — продажи по клиентам

**Цель:** топ покупателей, распределение по странам, средний чек.

In [13]:
mart_customers = spark.sql("""
    SELECT
        c.customer_id,
        CONCAT(c.first_name, ' ', c.last_name)       AS full_name,
        COALESCE(l.country, 'Unknown')               AS country,
        ROUND(CAST(SUM(f.total_price) AS DOUBLE), 2) AS total_purchases,
        ROUND(CAST(AVG(f.total_price) AS DOUBLE), 2) AS avg_check,
        COUNT(f.sale_id)                             AS purchase_count
    FROM fact_sales f
    JOIN  dim_customer c ON f.customer_id = c.customer_id
    LEFT JOIN dim_location l ON c.location_id = l.location_id
    GROUP BY c.customer_id, c.first_name, c.last_name, l.country
    ORDER BY total_purchases DESC
""")

mart_customers.show(5)

write_to_clickhouse(
    mart_customers,
    "mart_customers",
    """
    CREATE TABLE reports.mart_customers (
        customer_id     Int64,
        full_name       String,
        country         String,
        total_purchases Decimal(12, 2),
        avg_check       Decimal(12, 2),
        purchase_count  Int64
    ) ENGINE = MergeTree()
    ORDER BY customer_id
    """,
)

+-----------+-----------------+---------+---------------+---------+--------------+
|customer_id|        full_name|  country|total_purchases|avg_check|purchase_count|
+-----------+-----------------+---------+---------------+---------+--------------+
|       1025|    Gus Hartshorn|  Albania|         499.85|   499.85|             1|
|       8868|     Hayes McKain| Portugal|          499.8|    499.8|             1|
|       2851|        Ava Lomas|    China|         499.76|   499.76|             1|
|       8048|      Dawna Impey|Indonesia|         499.76|   499.76|             1|
|       7670|Lavinia Horsburgh|   Poland|         499.73|   499.73|             1|
+-----------+-----------------+---------+---------------+---------+--------------+
only showing top 5 rows
  ✓ reports.mart_customers: 10000 строк


---

## Витрина 3: mart_time — тренды продаж по времени

**Цель:** месячная и годовая динамика выручки и среднего чека.

In [14]:
mart_time = spark.sql("""
    SELECT
        YEAR(f.sale_date)                            AS year,
        MONTH(f.sale_date)                           AS month,
        ROUND(CAST(SUM(f.total_price) AS DOUBLE), 2) AS total_revenue,
        CAST(SUM(f.quantity) AS BIGINT)              AS total_quantity,
        ROUND(CAST(AVG(f.total_price) AS DOUBLE), 2) AS avg_order_size,
        COUNT(f.sale_id)                             AS order_count
    FROM fact_sales f
    WHERE f.sale_date IS NOT NULL
    GROUP BY YEAR(f.sale_date), MONTH(f.sale_date)
    ORDER BY year, month
""")

mart_time.show(10)

write_to_clickhouse(
    mart_time,
    "mart_time",
    """
    CREATE TABLE reports.mart_time (
        year           Int32,
        month          Int32,
        total_revenue  Decimal(18, 2),
        total_quantity Int64,
        avg_order_size Float64,
        order_count    Int64
    ) ENGINE = MergeTree()
    ORDER BY (year, month)
    """,
)

+----+-----+-------------+--------------+--------------+-----------+
|year|month|total_revenue|total_quantity|avg_order_size|order_count|
+----+-----+-------------+--------------+--------------+-----------+
|2021|    1|    224158.54|          4856|        256.47|        874|
|2021|    2|    192348.31|          4070|        260.28|        739|
|2021|    3|     207282.2|          4561|        245.89|        843|
|2021|    4|    206592.82|          4564|        246.83|        837|
|2021|    5|    211764.86|          4451|        255.75|        828|
|2021|    6|     215042.8|          4438|        261.61|        822|
|2021|    7|    220496.51|          4750|        256.99|        858|
|2021|    8|    221275.78|          4818|        246.68|        897|
|2021|    9|    210623.43|          4507|        251.04|        839|
|2021|   10|    228743.32|          4976|        256.44|        892|
+----+-----+-------------+--------------+--------------+-----------+
only showing top 10 rows
  ✓ repor

---

## Витрина 4: mart_stores — эффективность магазинов

**Цель:** топ-5 магазинов по выручке, распределение по городам и странам.

In [15]:
mart_stores = spark.sql("""
    SELECT
        s.store_id,
        s.store_name,
        COALESCE(l.city,    'Unknown') AS city,
        COALESCE(l.country, 'Unknown') AS country,
        ROUND(CAST(SUM(f.total_price) AS DOUBLE), 2) AS total_revenue,
        ROUND(CAST(AVG(f.total_price) AS DOUBLE), 2) AS avg_check,
        COUNT(f.sale_id)                             AS order_count
    FROM fact_sales f
    JOIN  dim_store    s ON f.store_id    = s.store_id
    LEFT JOIN dim_location l ON s.location_id = l.location_id
    GROUP BY s.store_id, s.store_name, l.city, l.country
    ORDER BY total_revenue DESC
""")

mart_stores.show(5)

write_to_clickhouse(
    mart_stores,
    "mart_stores",
    """
    CREATE TABLE reports.mart_stores (
        store_id      Int64,
        store_name    String,
        city          String,
        country       String,
        total_revenue Decimal(18, 2),
        avg_check     Decimal(12, 2),
        order_count   Int64
    ) ENGINE = MergeTree()
    ORDER BY store_id
    """,
)

+--------+-----------+---------+------------+-------------+---------+-----------+
|store_id| store_name|     city|     country|total_revenue|avg_check|order_count|
+--------+-----------+---------+------------+-------------+---------+-----------+
|    1405|       DabZ|   Grekan|South Africa|       499.85|   499.85|          1|
|    7824|Thoughtblab|    Fonte|      Poland|        499.8|    499.8|          1|
|    1145|     Camido|Longzhong|      Sweden|       499.76|   499.76|          1|
|    2188|   Edgeblab|    Pesek|   Indonesia|       499.76|   499.76|          1|
|    1263|    Centizu|   Tylicz|      Poland|       499.73|   499.73|          1|
+--------+-----------+---------+------------+-------------+---------+-----------+
only showing top 5 rows
  ✓ reports.mart_stores: 10000 строк


---

## Витрина 5: mart_suppliers — эффективность поставщиков

**Цель:** топ-5 поставщиков по выручке, средняя цена товаров, страны.

In [16]:
mart_suppliers = spark.sql("""
    SELECT
        sp.supplier_id,
        sp.supplier_name,
        COALESCE(l.country, 'Unknown')               AS country,
        ROUND(CAST(SUM(f.total_price) AS DOUBLE), 2) AS total_revenue,
        ROUND(CAST(AVG(p.price)       AS DOUBLE), 2) AS avg_product_price,
        COUNT(f.sale_id)                             AS order_count
    FROM fact_sales f
    JOIN  dim_product  p  ON f.product_id  = p.product_id
    JOIN  dim_supplier sp ON p.supplier_id = sp.supplier_id
    LEFT JOIN dim_location l ON sp.location_id = l.location_id
    GROUP BY sp.supplier_id, sp.supplier_name, l.country
    ORDER BY total_revenue DESC
""")

mart_suppliers.show(5)

write_to_clickhouse(
    mart_suppliers,
    "mart_suppliers",
    """
    CREATE TABLE reports.mart_suppliers (
        supplier_id       Int64,
        supplier_name     String,
        country           String,
        total_revenue     Decimal(18, 2),
        avg_product_price Decimal(12, 2),
        order_count       Int64
    ) ENGINE = MergeTree()
    ORDER BY supplier_id
    """,
)

+-----------+-------------+-----------+-------------+-----------------+-----------+
|supplier_id|supplier_name|    country|total_revenue|avg_product_price|order_count|
+-----------+-------------+-----------+-------------+-----------------+-----------+
|       5297|         Omba|     Poland|       862.83|             83.1|          2|
|       4564|        Meejo|       Iraq|       860.82|            39.65|          2|
|        847|    Browsecat|  Nicaragua|       819.92|            74.35|          2|
|       4189|         Layo|Philippines|       775.06|            44.16|          2|
|       9224|       Yambee|      China|       768.08|            40.72|          2|
+-----------+-------------+-----------+-------------+-----------------+-----------+
only showing top 5 rows
  ✓ reports.mart_suppliers: 9920 строк


---

## Витрина 6: mart_product_quality — качество продуктов

**Цель:** рейтинги, количество отзывов, корреляция рейтинга и продаж.

In [17]:
mart_product_quality = spark.sql("""
    SELECT
        p.product_id,
        p.product_name,
        ROUND(COALESCE(ANY_VALUE(p.rating), 0), 2) AS product_rating,
        COALESCE(ANY_VALUE(p.reviews), 0)        AS total_reviews,
        CAST(SUM(f.quantity) AS BIGINT)          AS total_sales,
        ROUND(CAST(SUM(f.total_price) AS DOUBLE), 2) AS total_revenue
    FROM fact_sales f
    JOIN dim_product p ON f.product_id = p.product_id
    GROUP BY p.product_id, p.product_name
    ORDER BY product_rating DESC
""")

mart_product_quality.show(5)

write_to_clickhouse(
    mart_product_quality,
    "mart_product_quality",
    """
    CREATE TABLE reports.mart_product_quality (
        product_id    Int64,
        product_name  String,
        product_rating Decimal(3, 1),
        total_reviews Int32,
        total_sales   Int64,
        total_revenue Decimal(18, 2)
    ) ENGINE = MergeTree()
    ORDER BY product_id
    """,
)

+----------+------------+--------------+-------------+-----------+-------------+
|product_id|product_name|product_rating|total_reviews|total_sales|total_revenue|
+----------+------------+--------------+-------------+-----------+-------------+
|      1343|   Bird Cage|           5.0|            8|          1|       103.67|
|      4150|     Cat Toy|           5.0|           41|         10|       484.47|
|      2039|   Bird Cage|           5.0|           17|          9|        79.27|
|       525|   Bird Cage|           5.0|          524|          3|        92.09|
|      2468|   Bird Cage|           5.0|          552|          2|       110.86|
+----------+------------+--------------+-------------+-----------+-------------+
only showing top 5 rows
  ✓ reports.mart_product_quality: 9920 строк


---

## Верификация — итоговые запросы

In [18]:
reports = [
    "mart_products",
    "mart_customers",
    "mart_time",
    "mart_stores",
    "mart_suppliers",
    "mart_product_quality",
]

print(f"{'Витрина':<25} {'Строк':>8}")
print("-" * 35)
for r in reports:
    n = ch.query(f"SELECT count() FROM reports.{r}").result_rows[0][0]
    print(f"{r:<25} {n:>8}")

Витрина                      Строк
-----------------------------------
mart_products                 9920
mart_customers               10000
mart_time                       12
mart_stores                  10000
mart_suppliers                9920
mart_product_quality          9920


In [19]:
# Топ-10 продуктов по выручке (проверка витрины 1)
top10 = ch.query(
    "SELECT product_name, category_name, total_revenue "
    "FROM reports.mart_products ORDER BY total_revenue DESC LIMIT 10"
).result_rows
print("Топ-10 продуктов по выручке:")
for row in top10:
    print(f"  {row[0]:<35} {row[1]:<15} {row[2]:>10.2f}")

Топ-10 продуктов по выручке:
  Dog Food                            Toy                 862.83
  Dog Food                            Toy                 860.82
  Cat Toy                             Toy                 819.92
  Bird Cage                           Toy                 775.06
  Cat Toy                             Toy                 768.08
  Bird Cage                           Cage                768.05
  Dog Food                            Food                760.92
  Dog Food                            Toy                 759.14
  Bird Cage                           Toy                 748.02
  Cat Toy                             Cage                747.86


In [20]:
# Месячные тренды продаж (проверка витрины 3)
trends = ch.query(
    "SELECT year, month, total_revenue, order_count "
    "FROM reports.mart_time ORDER BY year, month"
).result_rows
print("Месячные тренды:")
for row in trends:
    print(f"  {row[0]}-{row[1]:02d}  выручка={row[2]:>10.2f}  заказов={row[3]}")

Месячные тренды:
  2021-01  выручка= 224158.54  заказов=874
  2021-02  выручка= 192348.31  заказов=739
  2021-03  выручка= 207282.20  заказов=843
  2021-04  выручка= 206592.82  заказов=837
  2021-05  выручка= 211764.86  заказов=828
  2021-06  выручка= 215042.80  заказов=822
  2021-07  выручка= 220496.51  заказов=858
  2021-08  выручка= 221275.78  заказов=897
  2021-09  выручка= 210623.43  заказов=839
  2021-10  выручка= 228743.32  заказов=892
  2021-11  выручка= 200154.69  заказов=801
  2021-12  выручка= 191368.86  заказов=770


In [21]:
spark.stop()